# NeuroBridge-S4 Graph Learning — Phase 6: Within-Subject Longitudinal Graph Trajectories

> **Primary methodological direction:** NeuroBridge-S4 is primarily a within-subject
> longitudinal graph trajectory framework for small-N human research.

Phase 6 shifts the project from one-time reference comparison to **within-subject
longitudinal tracking**. Each subject is represented as a sequence of biological
adaptation graphs. The primary signal is change from the subject's own baseline.

> The primary signal is within-subject change from the individual's own baseline.
> Population reference data are used only to calibrate scale, estimate noise,
> contextualize rarity, and stabilize feature geometry.


## 1. Plain-language purpose

This notebook implements the central operational question for astronaut monitoring:

**How has this individual's biological adaptation graph changed relative to their
own baseline over time, across mission and recovery phases?**

Not: *Is this astronaut normal compared with healthy people?*

The conceptual model:

```text
participant + timepoint → biological adaptation graph
participant trajectory → sequence of biological adaptation graphs
trajectory signal → delta from personal baseline

G_subject_baseline → G_subject_mission → G_subject_postflight → G_subject_recovery
DeltaGraph(t) = Graph(t) - Graph(baseline)
```


## 2. Why one-time healthy-cohort comparison is insufficient

For astronaut monitoring, population-normal ranges are not enough. A value may remain
inside a healthy reference range while representing a meaningful deviation from that
astronaut's own baseline. Conversely, a stable individual baseline may differ from
generic norms without indicating a mission-relevant change.

> Healthy population ranges are insufficient for astronaut monitoring because
> operationally meaningful changes may occur within population-normal ranges, while
> stable individual baselines may differ from generic norms.

**Conceptual hierarchy:**

```text
Primary comparison:    Current astronaut state vs that astronaut's own baseline.
Secondary calibration: Reference cohort distribution for scale, noise, rarity, and feature geometry.
Interpretation layer:  NASA HRP five hazards as operational context, not exposure measurement.
```


## 3. Scientific guardrails

- This is **not** actual Artemis II data.
- If example data are used, they are **schema demonstration only** — not scientific evidence.
- The method does **not** diagnose.
- The method does **not** predict health outcomes.
- The method does **not** infer actual hazard exposure.
- HRP hazards are operational interpretation context.
- The **primary target** is within-subject change from personal baseline.


## Setup

In [1]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx

from neurobridge_graph.longitudinal import (
    detect_longitudinal_columns, validate_longitudinal_table,
    create_example_longitudinal_table, build_timepoint_graphs,
    compute_subject_trajectory_table, compute_longitudinal_delta_tables,
    sanitize_graph_for_graphml,
    compute_hazard_scores_per_timepoint,
    SCHEMA_DEMO_DATA_TYPE, PRIMARY_SIGNAL_STATEMENT, SELF_BASELINE_STATEMENT,
)
from neurobridge_graph.trajectory_features import (
    compute_hazard_context_delta, compute_recovery_metrics_table,
    identify_dominant_trajectory_shift,
)
from neurobridge_graph.trajectory_visualization import (
    plot_domain_delta_trajectory, plot_graph_metric_trajectory,
    plot_hazard_context_trajectory, plot_trajectory_heatmap,
    plot_recovery_summary,
)

RESULTS = repo_root / 'results'
TABLES = RESULTS / 'tables'
FIGURES = RESULTS / 'figures'
REPORTS = RESULTS / 'reports'
LONG_GRAPHS = RESULTS / 'graphs' / 'longitudinal'
for d in [TABLES, FIGURES, REPORTS, LONG_GRAPHS]:
    d.mkdir(parents=True, exist_ok=True)

print('Setup complete.')
print(PRIMARY_SIGNAL_STATEMENT)


Setup complete.
The primary signal is within-subject change from the individual's own baseline. Population reference data are used only to calibrate scale, estimate noise, contextualize rarity, and stabilize feature geometry.


## 5. Load or create longitudinal input

In [2]:
REAL_PATH = repo_root / 'data' / 'processed' / 'longitudinal_domain_scores.csv'
EXAMPLE_PATH = repo_root / 'data' / 'examples' / 'example_longitudinal_domain_scores.csv'

if REAL_PATH.exists():
    longitudinal_df = pd.read_csv(REAL_PATH)
    data_source = 'real_longitudinal_data'
    print(f'Loaded real longitudinal data: {REAL_PATH.relative_to(repo_root)}')
else:
    longitudinal_df = create_example_longitudinal_table(EXAMPLE_PATH)
    data_source = SCHEMA_DEMO_DATA_TYPE
    print(f'No real longitudinal data found.')
    print(f'Created schema demonstration table: {EXAMPLE_PATH.relative_to(repo_root)}')
    print()
    print('This demonstration uses a small example longitudinal table to test the')
    print('trajectory pipeline. It is NOT scientific evidence and does NOT represent')
    print('actual astronaut data.')

detected = detect_longitudinal_columns(longitudinal_df)
validation_report = validate_longitudinal_table(longitudinal_df)
validation_report.to_csv(TABLES / 'phase6_longitudinal_input_readiness_check.csv', index=False)
print(f'\nSubjects: {detected["n_subjects"]}, Timepoints: {detected["n_timepoints"]}')
print(f'Data source: {data_source}')
validation_report


No real longitudinal data found.
Created schema demonstration table: data/examples/example_longitudinal_domain_scores.csv

This demonstration uses a small example longitudinal table to test the
trajectory pipeline. It is NOT scientific evidence and does NOT represent
actual astronaut data.

Subjects: 2, Timepoints: 5
Data source: schema_demonstration_not_scientific_evidence


,check,status,detail
0,required_column:subject_id,ok,column present
1,required_column:timepoint,ok,column present
2,required_column:mission_phase,ok,column present
3,required_column:time_index,ok,column present
4,domain_columns,ok,6 numeric domain columns detected
5,n_subjects,ok,2
6,n_timepoints,ok,5
7,mission_phases,ok,"baseline, inflight, postflight, pre_mission, r..."
8,baseline_phase_present,ok,baseline phase found
9,data_type,ok,schema_demonstration_not_scientific_evidence


## 6. Longitudinal graph schema

Each row in the longitudinal table represents one **subject + timepoint**:

```text
subject_id + timepoint → biological adaptation graph
Graph(t) - Graph(baseline) → longitudinal adaptation signal
```

We compute:
- **node deltas** — per-domain activation change from personal baseline;
- **graph deltas** — graph-level metric changes;
- **hazard-context deltas** — HRP hazard relevance shifts;
- **recovery metrics** — recovery slope, recovery fraction, time-to-baseline-like state.


In [3]:
longitudinal_df.head()


,subject_id,timepoint,mission_phase,time_index,data_type,Cardiovascular regulation,Metabolic regulation,Body composition / physical status,Inflammation / immune-adjacent,Hematologic / oxygen-carrying,Recovery-related markers
0,Demo_Crew_01,T0_baseline,baseline,0,schema_demonstration_not_scientific_evidence,0.85,0.90,0.80,0.75,0.88,0.82
1,Demo_Crew_01,T1_pre,pre_mission,1,schema_demonstration_not_scientific_evidence,0.88,0.92,0.82,0.78,0.90,0.84
2,Demo_Crew_01,T2_inflight,inflight,2,schema_demonstration_not_scientific_evidence,1.10,1.05,1.20,1.15,1.08,0.95
3,Demo_Crew_01,T3_post,postflight,3,schema_demonstration_not_scientific_evidence,1.25,1.18,1.35,1.30,1.20,1.05
4,Demo_Crew_01,T4_recovery,recovery,4,schema_demonstration_not_scientific_evidence,0.95,0.98,0.92,0.88,0.95,0.86


## 7. Build timepoint graphs

In [4]:
graphs_by_subject = build_timepoint_graphs(longitudinal_df)

saved_graphml = []
for sid, tp_graphs in graphs_by_subject.items():
    for tp, G in tp_graphs.items():
        safe_sid = str(sid).replace(' ', '_')
        safe_tp = str(tp).replace(' ', '_')
        gpath = LONG_GRAPHS / f'subject_{safe_sid}__timepoint_{safe_tp}.graphml'
        nx.write_graphml(sanitize_graph_for_graphml(G), gpath)
        saved_graphml.append(gpath)

print(f'Built {sum(len(v) for v in graphs_by_subject.values())} timepoint graphs '
      f'for {len(graphs_by_subject)} subjects.')
print(f'Saved {len(saved_graphml)} GraphML files to results/graphs/longitudinal/')


Built 10 timepoint graphs for 2 subjects.
Saved 10 GraphML files to results/graphs/longitudinal/


## 8. Baseline-relative node deltas

In [5]:
delta_tables = compute_longitudinal_delta_tables(graphs_by_subject)
node_deltas = delta_tables['node_delta_table']
node_deltas.to_csv(TABLES / 'longitudinal_node_deltas.csv', index=False)
print('Saved -> results/tables/longitudinal_node_deltas.csv')
node_deltas.head(10)


Saved -> results/tables/longitudinal_node_deltas.csv


,subject_id,timepoint,mission_phase,time_index,data_type,domain,baseline_activation,current_activation,delta_activation,absolute_delta_activation,direction,activation_level_current
0,Demo_Crew_01,T0_baseline,baseline,0,schema_demonstration_not_scientific_evidence,Body composition / physical status,0.80,0.80,0.00,0.00,stable,mild
1,Demo_Crew_01,T0_baseline,baseline,0,schema_demonstration_not_scientific_evidence,Cardiovascular regulation,0.85,0.85,0.00,0.00,stable,mild
2,Demo_Crew_01,T0_baseline,baseline,0,schema_demonstration_not_scientific_evidence,Hematologic / oxygen-carrying,0.88,0.88,0.00,0.00,stable,mild
3,Demo_Crew_01,T0_baseline,baseline,0,schema_demonstration_not_scientific_evidence,Inflammation / immune-adjacent,0.75,0.75,0.00,0.00,stable,mild
4,Demo_Crew_01,T0_baseline,baseline,0,schema_demonstration_not_scientific_evidence,Metabolic regulation,0.90,0.90,0.00,0.00,stable,mild
5,Demo_Crew_01,T0_baseline,baseline,0,schema_demonstration_not_scientific_evidence,Recovery-related markers,0.82,0.82,0.00,0.00,stable,mild
6,Demo_Crew_01,T1_pre,pre_mission,1,schema_demonstration_not_scientific_evidence,Body composition / physical status,0.80,0.82,0.02,0.02,increase,mild
7,Demo_Crew_01,T1_pre,pre_mission,1,schema_demonstration_not_scientific_evidence,Cardiovascular regulation,0.85,0.88,0.03,0.03,increase,mild
8,Demo_Crew_01,T1_pre,pre_mission,1,schema_demonstration_not_scientific_evidence,Hematologic / oxygen-carrying,0.88,0.90,0.02,0.02,increase,mild
9,Demo_Crew_01,T1_pre,pre_mission,1,schema_demonstration_not_scientific_evidence,Inflammation / immune-adjacent,0.75,0.78,0.03,0.03,increase,mild


## 9. Graph-level deltas

In [6]:
graph_deltas = delta_tables['graph_delta_table']
graph_deltas.to_csv(TABLES / 'longitudinal_graph_deltas.csv', index=False)
print('Saved -> results/tables/longitudinal_graph_deltas.csv')
graph_deltas.head(10)


Saved -> results/tables/longitudinal_graph_deltas.csv


,subject_id,timepoint,mission_phase,time_index,metric,data_type,baseline_value,current_value,delta_value,absolute_delta_value
0,Demo_Crew_01,T0_baseline,baseline,0,mean_node_activation,schema_demonstration_not_scientific_evidence,0.83333,0.83333,0.00000,0.00000
1,Demo_Crew_01,T0_baseline,baseline,0,max_node_activation,schema_demonstration_not_scientific_evidence,0.90000,0.90000,0.00000,0.00000
2,Demo_Crew_01,T0_baseline,baseline,0,total_node_activation,schema_demonstration_not_scientific_evidence,5.00000,5.00000,0.00000,0.00000
3,Demo_Crew_01,T0_baseline,baseline,0,n_active_domains,schema_demonstration_not_scientific_evidence,0.00000,0.00000,0.00000,0.00000
4,Demo_Crew_01,T0_baseline,baseline,0,active_domain_fraction,schema_demonstration_not_scientific_evidence,0.00000,0.00000,0.00000,0.00000
5,Demo_Crew_01,T0_baseline,baseline,0,graph_density,schema_demonstration_not_scientific_evidence,0.26667,0.26667,0.00000,0.00000
6,Demo_Crew_01,T0_baseline,baseline,0,coactivation_edge_count,schema_demonstration_not_scientific_evidence,0.00000,0.00000,0.00000,0.00000
7,Demo_Crew_01,T1_pre,pre_mission,1,mean_node_activation,schema_demonstration_not_scientific_evidence,0.83333,0.85667,0.02334,0.02334
8,Demo_Crew_01,T1_pre,pre_mission,1,max_node_activation,schema_demonstration_not_scientific_evidence,0.90000,0.92000,0.02000,0.02000
9,Demo_Crew_01,T1_pre,pre_mission,1,total_node_activation,schema_demonstration_not_scientific_evidence,5.00000,5.14000,0.14000,0.14000


## 10. Hazard-context deltas

Hazard-context delta does **not** mean actual exposure. It indicates a shift in
biological graph pattern toward domains relevant to an HRP hazard context.


In [7]:
hazard_scores_long = compute_hazard_scores_per_timepoint(graphs_by_subject)

if not hazard_scores_long.empty:
    hazard_deltas = compute_hazard_context_delta(hazard_scores_long)
    hazard_deltas.to_csv(TABLES / 'longitudinal_hazard_deltas.csv', index=False)
    print('Saved -> results/tables/longitudinal_hazard_deltas.csv')
    hazard_deltas.head(8)
else:
    hazard_deltas = pd.DataFrame()
    print('No hazard-context deltas computed.')


Saved -> results/tables/longitudinal_hazard_deltas.csv


## 11. Recovery metrics

In [8]:
trajectory_table = compute_subject_trajectory_table(graphs_by_subject)
recovery_metrics = compute_recovery_metrics_table(trajectory_table)
recovery_metrics.to_csv(TABLES / 'recovery_metrics.csv', index=False)
print('Saved -> results/tables/recovery_metrics.csv')

trajectory_summary = identify_dominant_trajectory_shift(
    node_deltas, graph_deltas,
    hazard_deltas if not hazard_deltas.empty else None,
)
if not delta_tables['trajectory_summary'].empty:
    trajectory_summary = delta_tables['trajectory_summary'].merge(
        trajectory_summary, on=['subject_id', 'timepoint'], how='outer',
        suffixes=('', '_dominant'),
    )
trajectory_summary.to_csv(TABLES / 'longitudinal_trajectory_summary.csv', index=False)
print('Saved -> results/tables/longitudinal_trajectory_summary.csv')
recovery_metrics


Saved -> results/tables/recovery_metrics.csv
Saved -> results/tables/longitudinal_trajectory_summary.csv


,subject_id,metric,baseline_value,peak_value,final_value,peak_delta_from_baseline,final_delta_from_baseline,recovery_fraction,recovery_slope,time_to_baseline_like_state,interpretation
0,Demo_Crew_01,mean_node_activation,0.83333,1.22167,0.92333,0.38834,0.090,0.76824,None,1.0,mean_node_activation: strong recovery toward p...
1,Demo_Crew_01,max_node_activation,0.90000,1.35000,0.98000,0.45000,0.080,0.82222,None,1.0,max_node_activation: strong recovery toward pe...
2,Demo_Crew_01,total_node_activation,5.00000,7.33000,5.54000,2.33000,0.540,0.76824,None,1.0,total_node_activation: strong recovery toward ...
3,Demo_Crew_01,n_active_domains,0.00000,6.00000,0.00000,6.00000,0.000,1.00000,None,1.0,n_active_domains: strong recovery toward perso...
4,Demo_Crew_02,mean_node_activation,0.72833,1.06167,0.79333,0.33334,0.065,0.80500,None,1.0,mean_node_activation: strong recovery toward p...
5,Demo_Crew_02,max_node_activation,0.78000,1.20000,0.85000,0.42000,0.070,0.83333,None,1.0,max_node_activation: strong recovery toward pe...
6,Demo_Crew_02,total_node_activation,4.37000,6.37000,4.76000,2.00000,0.390,0.80500,None,1.0,total_node_activation: strong recovery toward ...
7,Demo_Crew_02,n_active_domains,0.00000,4.00000,0.00000,4.00000,0.000,1.00000,None,1.0,n_active_domains: strong recovery toward perso...


## 12. Visualize trajectories

In [9]:
plot_domain_delta_trajectory(
    node_deltas, output_path=FIGURES / 'phase6_domain_delta_trajectory.png')
plot_graph_metric_trajectory(
    trajectory_table, metric='mean_node_activation',
    output_path=FIGURES / 'phase6_graph_metric_trajectory.png')
if not hazard_deltas.empty:
    plot_hazard_context_trajectory(
        hazard_deltas, output_path=FIGURES / 'phase6_hazard_context_trajectory.png')
plot_trajectory_heatmap(
    node_deltas, output_path=FIGURES / 'phase6_trajectory_heatmap.png')
plot_recovery_summary(
    recovery_metrics, output_path=FIGURES / 'phase6_recovery_summary.png')
print('Saved 5 trajectory figures to results/figures/phase6_*.png')


Saved 5 trajectory figures to results/figures/phase6_*.png


## 13. Plain-language Phase 6 report

In [10]:
lines = []
lines.append('NeuroBridge-S4 Graph Learning — Phase 6 Report')
lines.append('Within-Subject Longitudinal Graph Trajectories')
lines.append('=' * 64)
lines.append('')
lines.append(PRIMARY_SIGNAL_STATEMENT)
lines.append('')
lines.append('Data source')
lines.append('-' * 64)
if data_source == SCHEMA_DEMO_DATA_TYPE:
    lines.append('  Schema demonstration / pipeline test data ONLY.')
    lines.append('  This is NOT scientific evidence and does NOT represent astronaut data.')
else:
    lines.append(f'  Real longitudinal data loaded from {REAL_PATH.name}')
lines.append('')
lines.append('Analysis summary')
lines.append('-' * 64)
n_subjects = longitudinal_df['subject_id'].nunique()
n_timepoints = longitudinal_df['timepoint'].nunique()
lines.append(f'  Subjects analyzed    : {n_subjects}')
lines.append(f'  Timepoints analyzed  : {n_timepoints}')
lines.append(f'  GraphML files saved  : {len(saved_graphml)}')
lines.append('  Baseline definition  : mission_phase == baseline, else earliest time_index')
lines.append('')
lines.append('Largest within-subject domain shifts')
lines.append('-' * 64)
if not node_deltas.empty:
    for sid in sorted(node_deltas['subject_id'].unique()):
        sub = node_deltas[node_deltas['subject_id'] == sid]
        top = sub.loc[sub['absolute_delta_activation'].idxmax()]
        lines.append(f'  {sid} @ {top["timepoint"]} ({top["mission_phase"]}): '
                     f'{top["domain"]} delta {top["delta_activation"]:+.2f}')
lines.append('')
lines.append('Largest graph-level shifts')
lines.append('-' * 64)
if not graph_deltas.empty:
    for sid in sorted(graph_deltas['subject_id'].unique()):
        sub = graph_deltas[graph_deltas['subject_id'] == sid]
        top = sub.loc[sub['absolute_delta_value'].idxmax()]
        lines.append(f'  {sid} @ {top["timepoint"]}: {top["metric"]} '
                     f'delta {top["delta_value"]:+.3f}')
lines.append('')
lines.append('Strongest hazard-context shifts')
lines.append('-' * 64)
if not hazard_deltas.empty:
    for sid in sorted(hazard_deltas['subject_id'].unique()):
        sub = hazard_deltas.dropna(subset=['delta_hazard_relevance'])
        if sub.empty: continue
        top = sub.loc[sub['delta_hazard_relevance'].abs().idxmax()]
        lines.append(f'  {sid} @ {top["timepoint"]}: {top["hazard"]} '
                     f'delta {top["delta_hazard_relevance"]:+.2f}')
lines.append('')
lines.append('Recovery observations')
lines.append('-' * 64)
for _, r in recovery_metrics.iterrows():
    frac = r.get('recovery_fraction')
    frac_str = f'{frac:.2f}' if pd.notna(frac) else 'n/a'
    lines.append(f'  {r["subject_id"]} / {r["metric"]}: recovery fraction {frac_str}')
lines.append('')
lines.append('Limitations')
lines.append('-' * 64)
lines.append('  - Self-baseline analysis requires longitudinal data across mission phases.')
lines.append('  - Schema demonstration data test the pipeline only; they are not evidence.')
lines.append('  - Hazard-context deltas are interpretation context, not exposure measurement.')
lines.append('  - n is very small; individualized trajectories avoid underpowered group inference.')
lines.append('')
lines.append('Why self-baseline matters')
lines.append('-' * 64)
lines.append('  ' + SELF_BASELINE_STATEMENT)
lines.append('  Reference cohorts remain useful for calibration but are not the main endpoint.')
lines.append('')
lines.append('Interpretation guardrail')
lines.append('-' * 64)
lines.append('  This report describes within-subject graph changes. It is not diagnosis,')
lines.append('  treatment guidance, exposure measurement, or health outcome prediction.')
lines.append('')
lines.append('Next phase')
lines.append('-' * 64)
lines.append('  Phase 7 — Explainable trajectory attribution: which subgraphs and domains')
lines.append('  drive the largest within-subject shifts across mission phases.')

report_text = '\n'.join(lines)
report_path = REPORTS / 'phase6_longitudinal_trajectory_report.txt'
report_path.write_text(report_text, encoding='utf-8')
print(f'Saved -> {report_path.relative_to(repo_root)}')
print()
print(report_text)


Saved -> results/reports/phase6_longitudinal_trajectory_report.txt

NeuroBridge-S4 Graph Learning — Phase 6 Report
Within-Subject Longitudinal Graph Trajectories

The primary signal is within-subject change from the individual's own baseline. Population reference data are used only to calibrate scale, estimate noise, contextualize rarity, and stabilize feature geometry.

Data source
----------------------------------------------------------------
  Schema demonstration / pipeline test data ONLY.
  This is NOT scientific evidence and does NOT represent astronaut data.

Analysis summary
----------------------------------------------------------------
  Subjects analyzed    : 2
  Timepoints analyzed  : 5
  GraphML files saved  : 10
  Baseline definition  : mission_phase == baseline, else earliest time_index

Largest within-subject domain shifts
----------------------------------------------------------------
  Demo_Crew_01 @ T3_post (postflight): Body composition / physical status delta +

## 14. Conclusion

Phase 6 makes NeuroBridge-S4 operationally more relevant to astronaut monitoring because it
focuses on **individual change over time** rather than one-time comparison to a generic
population. The primary signal is how each subject's biological adaptation graph shifts
relative to their own baseline across mission and recovery phases.

Phase 5 structural similarity remains useful as a demonstration, but **within-subject
longitudinal trajectory analysis** is the primary operationally relevant direction.
